# Bounded follow-up runner definitions

Purpose: reusable exact-resume factorial training and fixed epoch20 evaluation for the Notebook04 bounded follow-up. Inputs are an immutable external plan, the audited Food101 train/development bundle and a persisted absolute deadline. Outputs are per-arm checkpoints, initialization controls, paired trace certificates, frozen feature caches and across-training-seed comparisons. This definition notebook executes in seconds and starts no training by itself. Invoked training/evaluation uses the outer experiment's at-most-three-hour wall budget.

Checkpoints: `checkpoints/<run_id>/arms/<arm_id>/`, `preflight/` and `probes/`. Exact resume: open `notebooks/04_Bounded_Semantic_Followup.ipynb`, retain RUN_ID, plan and source definitions, and Run All. Resume uses the latest valid complete boundary and never resets the persisted deadline. Official Food101 test pixels remain unused.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)


## Plan binding and initialization

In [ ]:
"""Bounded multi-seed factorial training, exact recovery and fixed-final probes."""
from pathlib import Path
import copy, gc, hashlib, itertools, json, math, os, time
import numpy as np
import torch
from fmm_jepa.semantic_pilot import build_model, initialize_models, run_model, deadline_guard
from fmm_jepa.training import atomic_json, digest_file, definition_hashes, run_writer, write_csv
from fmm_jepa.food101_training import (training_protocol, validate_training_manifest,
    isolated_rng, _hash, _atomic_torch, _rng, _restore_rng)
from fmm_jepa.food101_evaluation import (encoder_state_hash, extract_features, fit_linear_probe,
    compute_diagnostics, stable_digest, standardize, classification_metrics, prediction_rows,
    _load_probe_checkpoint, atomic_torch)
from fmm_jepa.food101_data import EvaluationDataset, preprocessing_descriptor
from fmm_jepa.semantic_evaluation import _DeadlineDataset, _deadline_check, _release_semantic

RESUME_NOTEBOOK = "notebooks/04_Bounded_Semantic_Followup.ipynb"
RESUME_CELL = "launch-bounded-followup"

def followup_definition_hashes(root):
    root = Path(root)
    result = definition_hashes(root)
    for path in sorted((root / 'experiments/04_followup').glob('*.ipynb')):
        notebook = json.loads(path.read_text())
        cells = [''.join(c['source']) for c in notebook['cells']
                 if c['cell_type'] == 'code' and 'ahr-export' in c.get('metadata', {}).get('tags', [])]
        result[str(path.relative_to(root))] = hashlib.sha256('\n'.join(cells).encode()).hexdigest()
    return result

def _checked_plan(root, artifact_dir, plan, bundle):
    if not plan.get('arms') or plan.get('factor') not in ('ema', 'feature_rank'):
        raise ValueError('A nonempty EMA or feature-rank factorial plan is required')
    safe = lambda s: isinstance(s, str) and bool(s) and all(c.isalnum() or c in '-_' for c in s)
    if not safe(plan['run_id']) or any(not safe(a['id']) for a in plan['arms']):
        raise ValueError('Run and arm IDs must be safe path components')
    if len({a['id'] for a in plan['arms']}) != len(plan['arms']): raise ValueError('Duplicate arm IDs')
    if int(plan['updates']) < 4 or int(plan.get('round_updates', 100)) < 1:
        raise ValueError('Positive round size and at least four schedule updates required')
    if set(bundle['train_ids']) & set(bundle['dev_ids']): raise ValueError('Split overlap')
    if (set(bundle['train_ids']) | set(bundle['dev_ids'])) & set(bundle.get('heldout_ids', [])):
        raise ValueError('Official test overlaps training/development')
    for arm in plan['arms']:
        config = arm['config']
        if arm['kind'] not in ('vit', 'semantic'): raise ValueError('Unsupported architecture')
        if int(config['seeds']['model_init']) != int(arm['seed']): raise ValueError('Seed mismatch')
        if int(config['training']['updates']) != int(plan['updates']): raise ValueError('Unequal updates')
        if config.get('evaluation', {}).get('heldout_enabled', False): raise ValueError('Official test forbidden')
        if int(config['probe']['epochs']) != 20: raise ValueError('Registered final probe budget is 20 epochs')
        training_protocol(config)
    for seed in sorted({a['seed'] for a in plan['arms']}):
        group = [a for a in plan['arms'] if a['seed'] == seed]
        fields = ['data_order', 'augmentation', 'mask', 'probe']
        signatures = [{f: a['config']['seeds'][f] for f in fields} for a in group]
        if any(x != signatures[0] for x in signatures[1:]): raise ValueError('Within-seed random streams differ')
        batches = [training_protocol(a['config'])['effective_batch'] for a in group]
        if len(set(batches)) != 1: raise ValueError('Within-seed effective batch differs')
    sources = plan.get('source_hashes', followup_definition_hashes(root))
    actual = followup_definition_hashes(root)
    if any(actual.get(k) != v for k, v in sources.items()): raise ValueError('Registered export definitions changed')
    consumer_hash = plan.get('consumer_notebook_sha256')
    if consumer_hash:
        path = Path(root) / plan.get('resume_notebook', RESUME_NOTEBOOK)
        notebook = json.loads(path.read_text())
        code = [''.join(c['source']) for c in notebook['cells'] if c['cell_type'] == 'code']
        # Consumer uses the existing canonical JSON hash convention.
        if _hash(code) != consumer_hash: raise ValueError('Registered consumer code changed')
    return {'schema': 'fmm-jepa-followup-binding/v1', 'plan_hash': _hash(plan),
            'source_hashes': sources, 'dataset_fingerprint': bundle['dataset'].fingerprint,
            'split_sha256': bundle['split_manifest']['split_sha256']}

def _state_record(path):
    return {'path': str(Path(path).resolve()), 'sha256': digest_file(path)}

def _arm_paths(root, artifact, plan, arm):
    return (Path(artifact) / 'arms' / arm['id'],
            Path(root) / 'checkpoints' / plan['run_id'] / 'arms' / arm['id'])

def _resume_metadata(directory, plan, probe=False):
    path = Path(directory) / ('probe_manifest.json' if probe else 'manifest.json')
    if path.exists():
        value = json.loads(path.read_text())
        value['resume_notebook'] = plan.get('resume_notebook', RESUME_NOTEBOOK)
        value['resume_cell'] = plan.get('resume_cell', RESUME_CELL)
        value['resume_instructions'] = 'Run All with identical RUN_ID, plan and definitions; persisted deadline never resets.'
        atomic_json(path, value)

def _status(artifact, stage, state, **details):
    value = {'stage': stage, 'status': state, 'updated_unix': time.time(), **details}
    atomic_json(Path(artifact) / 'factorial_status.json', value)
    print(stage, state, details.get('arm_id', ''), flush=True)
    return value

def prepare_factorial(root, artifact_dir, plan, bundle):
    """Freeze per-arm initial states/protocols. CPU only; does not start training."""
    root, artifact = Path(root), Path(artifact_dir)
    artifact.mkdir(parents=True, exist_ok=True)
    binding = _checked_plan(root, artifact, plan, bundle)
    registration = artifact / 'factorial_registration.json'
    if registration.exists():
        saved = json.loads(registration.read_text())
        if saved['binding'] != binding: raise ValueError('Factorial registration changed')
    else:
        saved = {'binding': binding, 'arms': {}, 'heldout_status': 'NOT_RUN'}
    trainable_hashes = {}
    for arm in plan['arms']:
        folder, checkpoint = _arm_paths(root, artifact, plan, arm)
        folder.mkdir(parents=True, exist_ok=True)
        models, initial = initialize_models(arm['config'], folder)
        model = models[arm['kind']]
        params = {k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}
        parameter_hash = encoder_state_hash(params)
        key = int(arm['seed'])
        if key in trainable_hashes and trainable_hashes[key] != parameter_hash:
            raise AssertionError('ViT/semantic trainable initialization mismatch within seed')
        trainable_hashes[key] = parameter_hash
        protocol = {'schema': 'fmm-jepa-factorial-arm/v1', 'binding': binding,
                    'source_hashes': binding['source_hashes'], 'arm': arm,
                    'initialization': initial, 'training': training_protocol(arm['config']),
                    'selection': 'final registered successful-update budget; fixed epoch20 linear probe',
                    'heldout_status': 'NOT_RUN'}
        path = folder / 'protocol.json'
        if path.exists() and json.loads(path.read_text()) != protocol:
            raise ValueError('Arm protocol changed')
        atomic_json(path, protocol)
        saved['arms'][arm['id']] = {'protocol_sha256': digest_file(path),
             'initialization': initial, 'trainable_parameter_hash': parameter_hash,
             'checkpoint_directory': str(checkpoint.resolve())}
        del params, model, models
        gc.collect()
    atomic_json(registration, saved)
    return saved

def _registered(root, artifact, plan, bundle):
    binding = _checked_plan(root, artifact, plan, bundle)
    registration = json.loads((Path(artifact) / 'factorial_registration.json').read_text())
    if registration['binding'] != binding: raise ValueError('Factorial source/data/plan binding changed')
    for arm in plan['arms']:
        folder, _ = _arm_paths(root, artifact, plan, arm)
        if digest_file(folder / 'protocol.json') != registration['arms'][arm['id']]['protocol_sha256']:
            raise ValueError('Arm protocol checksum changed')
    return registration

def _initial_model(root, artifact, plan, arm):
    folder, _ = _arm_paths(root, artifact, plan, arm)
    info = json.loads((folder / 'initialization/manifest.json').read_text())
    record = info['states'][arm['kind']]['model']
    if digest_file(record['path']) != record['sha256']: raise ValueError('Initial model checksum changed')
    with isolated_rng(int(arm['seed'])): model = build_model(arm['config'], arm['kind']).cpu()
    model.load_state_dict(torch.load(record['path'], map_location='cpu', weights_only=True))
    return model


def _run_model_followup(plan, *args, **kwargs):
    """Correct resume metadata on every save without editing frozen upstream code."""
    import fmm_jepa.semantic_pilot as pilot_module
    original = pilot_module._save_state
    def save(*save_args, **save_kwargs):
        value = original(*save_args, **save_kwargs)
        _resume_metadata(save_args[0], plan)
        return json.loads((Path(save_args[0]) / 'manifest.json').read_text())
    pilot_module._save_state = save
    try:
        return run_model(*args, **kwargs)
    finally:
        pilot_module._save_state = original

def _fit_probe_followup(plan, *args, **kwargs):
    """Correct every probe-save handoff while preserving its checkpoint identity."""
    import fmm_jepa.food101_evaluation as evaluation_module
    original = evaluation_module._save_probe_checkpoint
    def save(folder, *save_args, **save_kwargs):
        value = original(folder, *save_args, **save_kwargs)
        _resume_metadata(folder, plan, probe=True)
        return value
    evaluation_module._save_probe_checkpoint = save
    try:
        return fit_linear_probe(*args, **kwargs)
    finally:
        evaluation_module._save_probe_checkpoint = original


## Full-state resume certificate

In [ ]:
def verify_factorial_resume(root, artifact_dir, plan, bundle, deadline_unix):
    """Four uninterrupted updates equal a fresh reconstruction after two updates.

    Test the actual chosen microbatch, accumulation and schedule for every
    architecture/factor level on the first registered seed. All trajectory RNG,
    model, optimizer, scheduler, sampler, EMA, traces and deterministic metrics
    must be bitwise equal. Timing is deliberately excluded.
    """
    from fmm_jepa.food101_benchmark import _food_exact_tree
    artifact, root = Path(artifact_dir), Path(root)
    registration = _registered(root, artifact, plan, bundle)
    path = artifact / 'factorial_preflight.json'
    if path.exists():
        certificate = json.loads(path.read_text())
        if certificate['binding'] != registration['binding'] or certificate['status'] != 'PASS':
            raise ValueError('Preflight binding/status invalid')
        for entry in certificate['arms'].values():
            for record in entry['checkpoints'].values():
                directory = Path(record['directory']); manifest = validate_training_manifest(directory)
                if manifest['successful_updates'] != 4 or digest_file(directory / manifest['latest']) != record['sha256']:
                    raise ValueError('Preflight evidence changed')
        return certificate
    selected, seen = [], set()
    for arm in plan['arms']:
        key = (arm['kind'], arm['condition'])
        if key not in seen: selected.append(arm); seen.add(key)
    fields = ['model','optimizer','scheduler','ema_position','scaler','precision','rng','sampler',
              'streams','microstep','partial_accumulation','world_size','rank','replay','config','protocol','identity']
    state_fields = ['successful_updates','attempted_updates','image_exposures','consecutive_failures',
                    'traces','failures','diagnostics']
    timing = {'data_seconds','compute_seconds','step_seconds','images_per_second','_cuda_event_count'}
    certificate = {'status': 'RUNNING', 'binding': registration['binding'], 'arms': {},
                   'scope': 'First seed per architecture/factor level; exact full state except elapsed timing'}
    original_rng = _rng()
    try:
        with deadline_guard(deadline_unix):
            for arm in selected:
                _deadline_check(deadline_unix, 5)
                _status(artifact, 'exact_resume', 'RUNNING', arm_id=arm['id'])
                arm_artifact, _ = _arm_paths(root, artifact, plan, arm)
                protocol = json.loads((arm_artifact / 'protocol.json').read_text())
                folder = artifact / 'preflight' / arm['id']; folder.mkdir(parents=True, exist_ok=True)
                trajectory_path = folder / 'trajectory_rng.pt'; rng_record = folder / 'trajectory_rng.json'
                if rng_record.exists():
                    if digest_file(trajectory_path) != json.loads(rng_record.read_text())['sha256']:
                        raise ValueError('Preflight trajectory RNG changed')
                    trajectory = torch.load(trajectory_path, map_location='cpu', weights_only=False)
                else:
                    if trajectory_path.exists(): raise ValueError('Orphan trajectory RNG')
                    with isolated_rng(int(arm['seed'])): trajectory = _rng()
                    _atomic_torch(trajectory_path, trajectory)
                    atomic_json(rng_record, _state_record(trajectory_path))
                ref = root / 'checkpoints' / plan['run_id'] / 'preflight' / (arm['id'] + '-reference')
                split = ref.with_name(arm['id'] + '-split')
                _restore_rng(copy.deepcopy(trajectory))
                reference = _run_model_followup(plan, arm['kind'], arm['config'], bundle, folder / 'reference', ref, protocol,
                    model=_initial_model(root, artifact, plan, arm), stop_after=4, deadline_unix=deadline_unix)
                del reference
                _restore_rng(copy.deepcopy(trajectory))
                partial = _run_model_followup(plan, arm['kind'], arm['config'], bundle, folder / 'split', split, protocol,
                    model=_initial_model(root, artifact, plan, arm), stop_after=2, deadline_unix=deadline_unix)
                if partial['resume']['final_step'] not in (2,4): raise AssertionError('Partial preflight incomplete')
                del partial
                resumed = _run_model_followup(plan, arm['kind'], arm['config'], bundle, folder / 'split', split, protocol,
                    model=_initial_model(root, artifact, plan, arm), stop_after=4, deadline_unix=deadline_unix)
                left_m, right_m = validate_training_manifest(ref), validate_training_manifest(split)
                left = torch.load(ref / left_m['latest'], map_location='cpu', weights_only=False)
                right = torch.load(split / right_m['latest'], map_location='cpu', weights_only=False)
                for key in fields: _food_exact_tree(left[key], right[key], key)
                for key in state_fields: _food_exact_tree(left['state'][key], right['state'][key], 'state.' + key)
                lm = [{k:v for k,v in row.items() if k not in timing} for row in left['state']['metrics']]
                rm = [{k:v for k,v in row.items() if k not in timing} for row in right['state']['metrics']]
                _food_exact_tree(lm, rm, 'deterministic_metrics')
                if left_m['successful_updates'] != 4 or right_m['attempted_updates'] != 4:
                    raise AssertionError('Preflight requires four successful finite updates')
                if len(right_m['checkpoints']) < 2: raise AssertionError('Latest-two retention missing')
                for directory in (ref, split): _resume_metadata(directory, plan)
                certificate['arms'][arm['id']] = {'status':'PASS','bitwise_equal':True,
                    'microbatch': arm['config']['training']['microbatch'],
                    'accumulation_steps': arm['config']['training']['accumulation_steps'],
                    'schedule_updates': plan['updates'], 'full_state_fields': fields,
                    'checkpoints': {'reference': {'directory':str(ref), 'sha256':digest_file(ref / left_m['latest'])},
                                    'resumed': {'directory':str(split), 'sha256':digest_file(split / right_m['latest'])}}}
                atomic_json(folder / 'certificate.json', certificate['arms'][arm['id']])
                del left, right, resumed
                _release_semantic(arm['config']['device'])
    finally: _restore_rng(original_rng)
    certificate['status'] = 'PASS'; atomic_json(path, certificate)
    return certificate


## Equal-budget training

In [ ]:
def run_factorial_training(root, artifact_dir, plan, bundle, deadline_unix):
    """Round-robin training with per-arm full-state recovery and one writer.

    The externally calibrated equal budget is immutable. A deadline cutoff is
    incomplete; it never becomes a post-hoc smaller confirmatory comparison.
    """
    if not os.environ.get('TMUX'): raise RuntimeError('Bounded long training must run in detached tmux')
    root, artifact = Path(root), Path(artifact_dir)
    registration = _registered(root, artifact, plan, bundle)
    certificate = json.loads((artifact / 'factorial_preflight.json').read_text())
    if certificate['status'] != 'PASS' or certificate['binding'] != registration['binding']:
        raise ValueError('Exact actual-config resume preflight required')
    training_deadline = float(deadline_unix) - float(plan.get('evaluation_reserve_seconds', 1800))
    budget, chunk = int(plan['updates']), int(plan.get('round_updates', 100))
    with run_writer(artifact / 'factorial_training_writer'):
        with deadline_guard(deadline_unix):
            for endpoint in range(chunk, budget + chunk, chunk):
                endpoint = min(endpoint, budget)
                counts = {}
                for arm in plan['arms']:
                    _, checkpoint = _arm_paths(root, artifact, plan, arm)
                    counts[arm['id']] = (validate_training_manifest(checkpoint)['successful_updates']
                        if (checkpoint / 'manifest.json').exists() else 0)
                if min(counts.values()) >= endpoint: continue
                pessimistic = float(plan.get('pessimistic_seconds_per_round_update', 0.0))
                remaining_round_estimate = pessimistic * (endpoint - min(counts.values()))
                if time.time() + remaining_round_estimate >= training_deadline:
                    _status(artifact, 'training_budget_exhausted', 'INCOMPLETE', steps=counts)
                    break
                for arm in plan['arms']:
                    if counts[arm['id']] >= endpoint: continue
                    _deadline_check(training_deadline)
                    folder, checkpoint = _arm_paths(root, artifact, plan, arm)
                    protocol = json.loads((folder / 'protocol.json').read_text())
                    _status(artifact, 'training', 'RUNNING', arm_id=arm['id'],
                            round_endpoint=endpoint, planned_updates=budget, completed_before_round=counts)
                    result = _run_model_followup(plan, arm['kind'], arm['config'], bundle, folder, checkpoint, protocol,
                        model=_initial_model(root, artifact, plan, arm), stop_after=endpoint,
                        deadline_unix=training_deadline)
                    _resume_metadata(checkpoint, plan)
                    final = int(result['resume']['final_step'])
                    del result
                    _release_semantic(arm['config']['device'])
                    if final != endpoint: break
                else: continue
                break
    records, seed_traces = {}, {}
    for arm in plan['arms']:
        folder, checkpoint = _arm_paths(root, artifact, plan, arm)
        if not (checkpoint / 'manifest.json').exists(): continue
        manifest = validate_training_manifest(checkpoint)
        payload = torch.load(checkpoint / manifest['latest'], map_location='cpu', weights_only=False)
        state = payload['state']
        fields = ('sample_ids', 'occurrences', 'augmentation_ids', 'mask_ids')
        traces = [{k: row[k] for k in fields} for row in state['traces']]
        trace_hash = _hash(traces); seed_traces.setdefault(arm['seed'], []).append(trace_hash)
        record = {'arm_id': arm['id'], 'kind':arm['kind'], 'condition':arm['condition'], 'seed':arm['seed'],
                  'successful_updates':state['successful_updates'], 'attempted_updates':state['attempted_updates'],
                  'image_exposures':state['image_exposures'], 'failures':state['failures'],
                  'paired_trace_sha256':trace_hash, 'checkpoint':_state_record(checkpoint / manifest['latest']),
                  'performance':json.loads((folder / (arm['kind'] + '_performance.json')).read_text())}
        records[arm['id']] = record
        del payload, state, traces
    steps = [r['successful_updates'] for r in records.values()]
    equal = len(steps) == len(plan['arms']) and len(set(steps)) == 1
    paired = equal and all(len(set(v)) == 1 for v in seed_traces.values())
    no_failures = all(not r['failures'] and r['attempted_updates'] == r['successful_updates'] for r in records.values())
    complete = paired and no_failures and steps[0] == budget
    summary = {'status':'COMPLETE' if complete else 'INCOMPLETE', 'binding':registration['binding'],
               'planned_updates':budget, 'equal_updates':equal, 'paired_traces_within_seed':paired,
               'no_failed_attempts':no_failures, 'arms':records, 'heldout_status':'NOT_RUN',
               'selection':'Every arm must finish the registered budget; no quality-based selection'}
    atomic_json(artifact / 'factorial_training_summary.json', summary)
    if not complete: raise RuntimeError('Factorial training incomplete or unpaired; no factorial quality conclusion allowed')
    return summary


## Fixed-final probes and seed comparisons

In [ ]:
def fit_fixed_final_probe(train_cache, dev_cache, checkpoint_root, config, model_id, encoder, deadline_unix, plan=None):
    """Use inherited exact-resume probe training, then evaluate its final epoch.

    The inherited helper also emits a best-dev artifact; that artifact is never
    used here. The authoritative final-epoch classifier is in the latest fully
    completed resumable checkpoint, saved before best-dev selection occurs.
    """
    plan = plan or {}
    folder = Path(checkpoint_root) / model_id
    chunk = int(config.get('evaluation', {}).get('probe_deadline_chunk_updates', 50))
    if not 1 <= chunk <= 250: raise ValueError('Probe chunk must be at most250 updates')
    while True:
        _deadline_check(deadline_unix, 2)
        path = folder / 'probe_manifest.json'
        updates = int(json.loads(path.read_text())['latest']['successful_updates']) if path.exists() else 0
        result = _fit_probe_followup(plan, train_cache, dev_cache, checkpoint_root, config, model_id,
                                  encoder=encoder, stop_after_updates=updates + chunk)
        _resume_metadata(folder, plan, probe=True)
        if result.get('status') != 'INTERRUPTED': break
    saved = _load_probe_checkpoint(folder, result['selected_probe']['identity'])
    if saved['completed_epochs'] != 20 or saved['active_order'] is not None or saved['next_offset'] != 0:
        raise AssertionError('Final probe checkpoint is not a completed fixed epoch20')
    if saved['initialization_hash'] != result['summary']['initialization_hash']:
        raise AssertionError('Probe initialization evidence mismatch')
    # CPU reconstruction is independent of inherited best-dev predictions.
    classifier = torch.nn.Linear(train_cache['features'].shape[1], 101)
    classifier.load_state_dict(saved['classifier']); classifier.eval()
    standardized = standardize(dev_cache['features'], saved['standardization'])
    with torch.inference_mode(): logits = torch.cat([classifier(part) for part in standardized.split(256)])
    metrics = classification_metrics(logits, dev_cache['labels'])
    final_metrics = [x for x in saved['metrics'] if x['split'] == 'development' and x['epoch'] == 20]
    if len(final_metrics) != 1 or abs(metrics['top1'] - final_metrics[0]['top1']) > 1e-7:
        raise AssertionError('Independent final classifier reconstruction disagrees with epoch20 top1')
    probe_hash = encoder_state_hash(classifier)
    predictions = prediction_rows(logits, dev_cache['labels'], dev_cache['sample_ids'], model_id,
        'development', train_cache['identity']['checkpoint_hash'], probe_hash, result['summary']['seed'])
    selected = {'state_dict': saved['classifier'], 'standardization':saved['standardization'],
                'config':{**result['summary']['budget'], 'selection':'fixed_final_epoch'},
                'selected_epoch':20, 'probe_hash':probe_hash,
                'encoder_checkpoint_hash':train_cache['identity']['checkpoint_hash'],
                'identity':result['selected_probe']['identity'], 'selection':'fixed_final_epoch',
                'source_resume_manifest':str(folder / 'probe_manifest.json')}
    atomic_torch(folder / 'final_epoch_probe.pt', selected)
    write_csv(folder / 'final_epoch_predictions.csv', predictions)
    summary = {**result['summary'], 'selected_epoch':20, 'selection':'fixed_final_epoch',
               'probe_hash':probe_hash, 'metrics':metrics, 'budget':selected['config'],
               'selected_probe_path':str(folder / 'final_epoch_probe.pt'),
               'predictions_path':str(folder / 'final_epoch_predictions.csv'),
               'independent_final_prediction_reconstruction':'PASS',
               'historical_best_dev_artifact_used':False}
    atomic_json(folder / 'final_epoch_summary.json', summary)
    return {'summary':summary, 'metrics':result['metrics'], 'predictions':predictions,
            'performance':result['performance'], 'selected_probe':selected}

def _seed_statistics(values):
    from scipy.stats import t as student_t
    values = np.asarray(values, dtype=float); n = len(values)
    mean = float(values.mean()); sd = float(values.std(ddof=1)) if n > 1 else None
    half = float(student_t.ppf(.975, n-1) * sd / math.sqrt(n)) if n > 1 else None
    return {'n_training_seeds':n, 'mean_percentage_points':mean, 'sd_percentage_points':sd,
            'range_percentage_points':[float(values.min()),float(values.max())],
            'approximate_t_ci95_percentage_points':[mean-half,mean+half] if half is not None else None,
            'positive_seeds':int((values > 0).sum()), 'negative_seeds':int((values < 0).sum()),
            'uncertainty_scope':'Across training seeds conditional on this fixed development set; t interval is fragile with three seeds. No multiplicity adjustment.'}

def _factorial_contrasts(plan, evaluated, arm_controls):
    per_seed = []
    def score(model_id):
        return float(np.mean([p['correct_top1'] for p in evaluated[model_id]['probe']['predictions']])) * 100
    def add(label, seed, first, second):
        a, b = evaluated[first]['probe']['predictions'], evaluated[second]['probe']['predictions']
        amap, bmap = {p['sample_id']:p for p in a}, {p['sample_id']:p for p in b}
        if set(amap) != set(bmap) or any(amap[k]['label'] != bmap[k]['label'] for k in amap):
            raise AssertionError('Contrast memberships or labels differ')
        delta = float(np.mean([amap[k]['correct_top1']-bmap[k]['correct_top1'] for k in sorted(amap)]))*100
        per_seed.append({'contrast':label, 'seed':int(seed), 'first':first, 'second':second,
                         'first_top1_percent':score(first), 'second_top1_percent':score(second),
                         'delta_percentage_points':delta, 'development_examples':len(amap)})
    for arm in plan['arms']:
        add(arm['kind'] + ':' + arm['condition'] + ' trained minus initialization', arm['seed'],
            arm['id'], arm_controls[arm['id']])
    for seed in sorted({a['seed'] for a in plan['arms']}):
        arms = [a for a in plan['arms'] if a['seed'] == seed]
        for first, second in itertools.combinations(arms, 2):
            # All pairwise contrasts are explicitly exploratory, not selected by score.
            label = first['kind']+':'+first['condition']+' minus '+second['kind']+':'+second['condition']
            add(label, seed, first['id'], second['id'])
    aggregate = {}
    for label in dict.fromkeys(row['contrast'] for row in per_seed):
        rows = [r for r in per_seed if r['contrast'] == label]
        aggregate[label] = {**_seed_statistics([r['delta_percentage_points'] for r in rows]),
                            'paired_seed_deltas':[{k:r[k] for k in ('seed','delta_percentage_points')} for r in rows]}
    return {'per_seed':per_seed, 'across_seed':aggregate,
            'interpretation':'Exploratory paired training-seed contrasts. Individual image bootstrap is not substituted for seed variability; multiple contrasts are not confirmatory tests.'}


## Frozen representation evaluation

In [ ]:
def write_factorial_report(artifact_dir, plan, evaluation, training):
    """Every seed remains visible; axes include every measured value."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    artifact=Path(artifact_dir); ev=evaluation['evaluations']; controls=evaluation['arm_initialization_controls']; rows=[]
    for arm in plan['arms']:
        perf=training['arms'][arm['id']]['performance'];step=perf.get('complete_jepa_step_seconds');peak=perf.get('peak_cuda_allocated_bytes')
        initial=100*ev[controls[arm['id']]]['probe']['summary']['metrics']['top1'];trained=100*ev[arm['id']]['probe']['summary']['metrics']['top1']
        rows.append({'arm_id':arm['id'],'kind':arm['kind'],'condition':arm['condition'],'seed':arm['seed'],
                     'initial_top1_percent':initial,'trained_top1_percent':trained,'gain_percentage_points':trained-initial,
                     'median_training_step_seconds':None if step is None else step['median'],
                     'training_peak_GiB':None if peak is None else peak/1024**3})
    chart_path=artifact/'factorial_chart_data.json';atomic_json(chart_path,rows)
    categories=list(dict.fromkeys((r['kind'],r['condition']) for r in rows));seeds=sorted({r['seed'] for r in rows})
    labels=[('ViT' if k=='vit' else 'Semantic')+'\n'+c.replace('_',' ') for k,c in categories]
    colors=['#2966A0','#A26822','#5D7146'];markers=['o','s','^']
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})
    fig,ax=plt.subplots(figsize=(max(8,2.5*len(categories)),5));all_values=[]
    for index,category in enumerate(categories):
        group=[r for r in rows if (r['kind'],r['condition'])==category]
        for si,seed in enumerate(seeds):
            row=next(r for r in group if r['seed']==seed);offset=(si-(len(seeds)-1)/2)*.045
            xs=[index-.20+offset,index+.20+offset];ys=[row['initial_top1_percent'],row['trained_top1_percent']];all_values.extend(ys)
            ax.plot(xs,ys,color=colors[si%3],alpha=.5,lw=1.3)
            ax.scatter(xs[0],ys[0],marker=markers[si%3],facecolor='white',edgecolor=colors[si%3],s=52,zorder=3)
            ax.scatter(xs[1],ys[1],marker=markers[si%3],color=colors[si%3],s=52,zorder=3,label='Seed '+str(seed) if index==0 else None)
        for offset,name in [(-.20,'Initial'),(.20,'Trained')]:ax.text(index+offset,.03,name,ha='center',fontsize=9,transform=ax.get_xaxis_transform())
    ax.set_xticks(range(len(categories)),labels);ax.set_ylabel('Development top-1 accuracy (%)')
    ax.set_title('Fixed epoch20 probes: initialization and final EMA encoder',loc='left',pad=18)
    ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True);ax.set_ylim(0,max(all_values)*1.25);ax.set_xlim(-.55,len(categories)-.45)
    ax.legend(frameon=False,loc='upper left',ncol=min(3,len(seeds)),fontsize=9)
    fig.text(.01,.015,'Food101: 2,020 fixed development images. Points: paired training seeds. Open = initial; filled = trained.',fontsize=9)
    fig.tight_layout(rect=(0,.07,1,1));quality_path=artifact/'factorial_quality.png';fig.savefig(quality_path,dpi=160);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,max(4,1.1*len(categories))))
    for ax,field,title,unit in [(axes[0],'median_training_step_seconds','Actual accumulated training step','Seconds per update'),(axes[1],'training_peak_GiB','Peak allocated training memory','GiB')]:
        all_values=[]
        for index,category in enumerate(categories):
            group=[r for r in rows if (r['kind'],r['condition'])==category];values=[r[field] for r in group if r[field] is not None]
            if not values:continue
            mean=float(np.mean(values));all_values.extend(values)
            ax.barh(index,mean,color='#2966A0' if category[0]=='vit' else '#5D7146',alpha=.30,height=.48)
            for si,seed in enumerate(seeds):
                row=next(r for r in group if r['seed']==seed)
                if row[field] is not None:ax.scatter(row[field],index+(si-(len(seeds)-1)/2)*.08,marker=markers[si%3],color=colors[si%3],s=35,zorder=3)
            ax.text(mean,index-.29,f'{mean:.3f}',ha='center',va='bottom',fontsize=9)
        ax.set_yticks(range(len(categories)),labels);ax.invert_yaxis();ax.set_xlabel(unit);ax.set_title(title,loc='left')
        ax.set_xlim(0,max(all_values)*1.22 if all_values else 1);ax.grid(axis='x',alpha=.18);ax.set_axisbelow(True)
    fig.text(.01,.012,'Bars: mean across seed runs; points: individual runs. Step includes data preparation; memory excludes CUDA context/reserved cache.',fontsize=9)
    fig.tight_layout(rect=(0,.065,1,1));compute_path=artifact/'factorial_compute.png';fig.savefig(compute_path,dpi=160);plt.close(fig)
    manifest_path=artifact/'factorial_figure_manifest.json'
    atomic_json(manifest_path,{'source':str(chart_path),'source_sha256':digest_file(chart_path),
        'figures':[str(quality_path),str(compute_path)],'visual_inspection':'PENDING',
        'axis_check':'Zero-based absolute axes; limits set after all series','uncertainty':'All training seeds shown; bars are means, not confidence bounds'})
    lines=['# Bounded semantic follow-up results','',f"Factor: **{plan['factor']}**. **{len(seeds)} paired training seeds**, **{plan['updates']} successful updates per arm**, fixed epoch20 linear probes.",'',
      'Official Food101 test remains unused. Development membership was reused from earlier exploratory work.','',
      '| Architecture / condition | Initial top-1 mean | Trained top-1 mean | Mean gain | Step mean | Peak mean |','|---|---:|---:|---:|---:|---:|']
    for category in categories:
        group=[r for r in rows if (r['kind'],r['condition'])==category]
        mean=lambda field:float(np.mean([r[field] for r in group if r[field] is not None]))
        lines.append(f"| {category[0]} / {category[1]} | {mean('initial_top1_percent'):.2f}% | {mean('trained_top1_percent'):.2f}% | {mean('gain_percentage_points'):+.2f} pp | {mean('median_training_step_seconds'):.3f} s | {mean('training_peak_GiB'):.3f} GiB |")
    lines+=['','Accuracy averages seed-specific scores. Step averages each run median; memory averages run peaks.','',
      '![Accuracy and initialization controls](factorial_quality.png)','','![Actual training compute](factorial_compute.png)','',
      '## Paired seed contrasts','','| Contrast | Individual seed deltas (pp) | Mean (pp) | Approximate across-seed 95% t interval (pp) |','|---|---|---:|---|']
    for label,value in evaluation['contrasts']['across_seed'].items():
        deltas=', '.join(f"{r['seed']}: {r['delta_percentage_points']:+.2f}" for r in value['paired_seed_deltas']);ci=value['approximate_t_ci95_percentage_points']
        interval='Not estimated' if ci is None else f'[{ci[0]:+.2f}, {ci[1]:+.2f}]'
        lines.append(f"| {label} | {deltas} | {value['mean_percentage_points']:+.2f} | {interval} |")
    lines+=['','Three seeds provide a variance screen; t intervals are fragile at this sample size and conditional on the fixed development set. Comparisons are exploratory and unadjusted for multiplicity. Intervals crossing zero do not establish equivalence or noninferiority.','',
      'Every probe uses the fixed final epoch20 classifier. Historical helper best-development artifacts are excluded. Final predictions were reconstructed independently from saved final classifier weights and train-only standardization.','',
      'Required gates: actual-config full-state checkpoint recovery, identical within-seed data/augmentation/mask traces, equal updates, no failed attempts and unchanged source bindings.','',
      'Machine-readable evidence: `factorial_evaluation.json`, `factorial_training_summary.json`, `factorial_chart_data.json` and per-model predictions in `predictions/`.']
    report_path=artifact/'factorial_results.md';temp=report_path.with_suffix('.md.tmp');temp.write_text('\n'.join(lines)+'\n');temp.replace(report_path)
    return {'report_path':str(report_path),'figure_paths':[str(quality_path),str(compute_path)],'figure_manifest_path':str(manifest_path),'chart_data_path':str(chart_path)}


In [ ]:
def evaluate_factorial(root, artifact_dir, plan, bundle, deadline_unix):
    """Evaluate fixed final EMA checkpoints plus distinct initialization controls."""
    root, artifact = Path(root), Path(artifact_dir)
    registration = _registered(root, artifact, plan, bundle)
    training = json.loads((artifact / 'factorial_training_summary.json').read_text())
    if training['status'] != 'COMPLETE' or training['binding'] != registration['binding']:
        raise ValueError('Only complete equal-budget factorial training may be compared')
    controls, jobs, arm_controls = {}, [], {}
    for arm in plan['arms']:
        initial = registration['arms'][arm['id']]['initialization']['states'][arm['kind']]['teacher']
        key = (arm['kind'], arm['seed'], initial['sha256'])
        if key not in controls:
            control_id = arm['kind'] + '-seed' + str(arm['seed']) + '-initial-' + initial['sha256'][:10]
            controls[key] = control_id
            jobs.append({'model_id':control_id, 'arm':arm, 'control':'initialization', 'state':initial})
        arm_controls[arm['id']] = controls[key]
    jobs.extend({'model_id':arm['id'], 'arm':arm, 'control':'trained',
                 'state':training['arms'][arm['id']]['checkpoint']} for arm in plan['arms'])
    identity = {'binding':registration['binding'], 'jobs':jobs, 'probe_selection':'fixed_final_epoch20',
                'train_ids':bundle['train_ids'], 'development_ids':bundle['dev_ids'], 'heldout_status':'NOT_RUN'}
    identity_hash = stable_digest(identity)
    path = artifact / 'factorial_evaluation_manifest.json'
    if path.exists():
        manifest = json.loads(path.read_text())
        if manifest['identity_hash'] != identity_hash: raise ValueError('Evaluation identity changed')
    else:
        manifest = {'identity_hash':identity_hash, 'identity':identity, 'completed':{}}
        atomic_json(path, manifest)
    evaluated = {}; started = time.time()
    with run_writer(artifact / 'factorial_evaluation_writer'):
        with deadline_guard(deadline_unix):
            for job in jobs:
                model_id, arm = job['model_id'], job['arm']; config = arm['config']
                summary_path = artifact / 'evaluation' / (model_id + '.json')
                if model_id in manifest['completed']:
                    if digest_file(summary_path) != manifest['completed'][model_id]['sha256']:
                        raise ValueError('Completed evaluation evidence changed')
                    evaluated[model_id] = json.loads(summary_path.read_text())
                    continue
                _deadline_check(deadline_unix, 5)
                _status(artifact, 'evaluation', 'RUNNING', arm_id=model_id,
                        completed_probes=len(evaluated), total_probes=len(jobs))
                if digest_file(job['state']['path']) != job['state']['sha256']:
                    raise ValueError('Encoder state checksum changed')
                model = build_model(config, arm['kind']).cpu()
                state = torch.load(job['state']['path'], map_location='cpu', weights_only=job['control']=='initialization')
                if job['control'] == 'initialization': model.target.load_state_dict(state)
                else: model.load_state_dict(state['model'])
                del state
                encoder = model.target; del model
                common = {'dataset_revision':bundle['dataset'].fingerprint,
                          'checkpoint_hash':encoder_state_hash(encoder), 'encoder_kind':arm['kind'],
                          'preprocessing':preprocessing_descriptor(config),
                          'pooling':'mean_final_layernorm_patches',
                          'normalization':'encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization'}
                caches = {}; performance = []
                for split, ids in (('train',bundle['train_ids']),('development',bundle['dev_ids'])):
                    dataset = _DeadlineDataset(EvaluationDataset(bundle['dataset'], ids, config), deadline_unix)
                    cache = extract_features(encoder, dataset, artifact / 'features' / model_id,
                        {**common,'sample_ids':list(ids)}, batch_size=int(config['evaluation']['batch_size']),
                        device=config['device'])
                    caches[split] = cache; performance.append({'split':split, **cache['performance']})
                encoder.to(config['device'])
                diagnostic_data = _DeadlineDataset(EvaluationDataset(bundle['dataset'],bundle['diagnostic_ids'],config), deadline_unix)
                diagnostics = compute_diagnostics(encoder, diagnostic_data, config)
                encoder.cpu(); _release_semantic(config['device'])
                result = fit_fixed_final_probe(caches['train'], caches['development'],
                    root / 'checkpoints' / plan['run_id'] / 'probes', config, model_id, encoder, deadline_unix, plan)
                saved = {'identity_hash':identity_hash, 'model_id':model_id, 'kind':arm['kind'],
                         'condition':arm['condition'], 'seed':arm['seed'], 'control':job['control'],
                         'source_encoder':job['state'], 'feature_caches':{k:_state_record(v['path']) for k,v in caches.items()},
                         'probe':{k:result[k] for k in ('summary','metrics','predictions')},
                         'diagnostics':diagnostics, 'performance':performance,
                         'probe_performance_last_chunk':result['performance']}
                write_csv(artifact / 'predictions' / (model_id + '.csv'), result['predictions'])
                atomic_json(summary_path,saved); evaluated[model_id] = saved
                manifest['completed'][model_id] = _state_record(summary_path); atomic_json(path,manifest)
                print(model_id, 'fixed epoch20 top1', 100*result['summary']['metrics']['top1'], flush=True)
                del encoder, caches, result
                _release_semantic(config['device'])
    # Check identical classifier initialization and budget for matched model seeds.
    for seed in sorted({a['seed'] for a in plan['arms']}):
        group = [v for v in evaluated.values() if v['seed'] == seed]
        if len({v['probe']['summary']['initialization_hash'] for v in group}) != 1:
            raise AssertionError('Within-seed probe initialization mismatch')
        if len({_hash(v['probe']['summary']['budget']) for v in group}) != 1:
            raise AssertionError('Within-seed probe budget mismatch')
    contrasts = _factorial_contrasts(plan, evaluated, arm_controls)
    result = {'schema':'fmm-jepa-factorial-evaluation/v1', 'status':'COMPLETE',
              'identity_hash':identity_hash, 'binding':registration['binding'],
              'arm_initialization_controls':arm_controls, 'evaluations':evaluated,
              'contrasts':contrasts, 'wall_seconds_this_invocation':time.time()-started,
              'probe_selection':'fixed_final_epoch20; historical helper best-dev outputs not used',
              'heldout_status':'NOT_RUN', 'training_seed_count':len({a['seed'] for a in plan['arms']}),
              'scope':'Short multi-seed promise test on fixed reused development data; neither test-set generalization nor a scaling law.'}
    result.update(write_factorial_report(artifact, plan, result, training))
    atomic_json(artifact / 'factorial_evaluation.json', result)
    _status(artifact, 'evaluation', 'COMPLETED', evaluated_models=len(evaluated))
    return result


## CPU API verification

This final non-export cell runs a small CPU fixture: both architectures and EMA settings pass full-state recovery; a synthetic probe verifies fixed epoch20 selection. Fixtures are explicitly excluded from quality evidence. Expected runtime: under one minute.

In [ ]:
# Bounded CPU-only API smoke: no GPU training, no production artifact changes.
import uuid
from fmm_jepa.semantic_pilot import reuse_food_bundle
from fmm_jepa.food101_evaluation import FEATURE_SCHEMA
_cpu_run = 'followup-runner-cpu-' + uuid.uuid4().hex[:10]
_cpu_artifact = ROOT / 'artifacts/notebook_04/implementation_review' / _cpu_run
_cpu_config = json.loads((ROOT / 'configs/semantic_local_pilot.yaml').read_text())
_cpu_config.update(device='cpu', cpu_threads=2, image_size=32)
_cpu_config['model'].update(dim=24, depth=1, heads=3, predictor_dim=16, predictor_depth=1,
                            predictor_heads=4, semantic_features=8)
_cpu_config['training'].update(updates=4, microbatch=2, accumulation_steps=1, checkpoint_every=1)
_cpu_config['seeds']['model_init']=20260926
_cpu_config['evaluation'].update(batch_size=2, diagnostic_batch_size=2)
_cpu_config['probe'].update(batch_size=256, epochs=20, device='cpu')
torch.set_num_threads(2);torch.use_deterministic_algorithms(True)
_cpu_bundle = reuse_food_bundle(ROOT, _cpu_config, verify_images=False)
_cpu_plan={'run_id':_cpu_run,'factor':'ema','updates':4,'round_updates':2,'seeds':[20260926],
           'resume_notebook':RESUME_NOTEBOOK,'resume_cell':RESUME_CELL,'arms':[]}
for _kind in ('vit','semantic'):
    for _condition,_momentum in [('base',.996),('fast',.99)]:
        _cfg=copy.deepcopy(_cpu_config);_cfg['training']['ema_start']=_momentum
        _cpu_plan['arms'].append({'id':_kind+'-'+_condition,'kind':_kind,'seed':20260926,
                                  'condition':_condition,'config':_cfg})
prepare_factorial(ROOT,_cpu_artifact,_cpu_plan,_cpu_bundle)
_cpu_certificate=verify_factorial_resume(ROOT,_cpu_artifact,_cpu_plan,_cpu_bundle,time.time()+120)
assert _cpu_certificate['status']=='PASS' and len(_cpu_certificate['arms'])==4
for _entry in _cpu_certificate['arms'].values():
    for _record in _entry['checkpoints'].values():
        _manifest=json.loads((Path(_record['directory'])/'manifest.json').read_text())
        assert _manifest['resume_notebook']==RESUME_NOTEBOOK and _manifest['resume_cell']==RESUME_CELL
# Deterministic synthetic features are used only to check probe recovery and selection.
_generator=torch.Generator().manual_seed(7171)
def _cpu_cache(prefix,count):
    features=torch.randn(count,24,generator=_generator);labels=torch.arange(count)%101
    identity={'dataset_revision':'SYNTHETIC_CPU_QA','checkpoint_hash':'SYNTHETIC_CPU_QA',
      'encoder_kind':'vit','preprocessing':{'fixture':True},'sample_ids':[prefix+str(i) for i in range(count)],
      'pooling':'fixture','normalization':'fixture'}
    return {'schema':FEATURE_SCHEMA,'identity':identity,'features':features,'labels':labels,
            'sample_ids':identity['sample_ids'],'content_hash':encoder_state_hash({'features':features,'labels':labels})}
_cpu_probe=fit_fixed_final_probe(_cpu_cache('train',202),_cpu_cache('dev',101),
    ROOT/'checkpoints'/_cpu_run/'probes',_cpu_config,'synthetic-cpu-probe',None,time.time()+60,_cpu_plan)
assert _cpu_probe['summary']['selected_epoch']==20
assert _cpu_probe['summary']['independent_final_prediction_reconstruction']=='PASS'
_cpu_summary={'status':'PASS','cpu_resume_architecture_conditions':4,'fixed_final_probe':'PASS',
              'synthetic_probe_fixture':True,'production_quality_evidence':False,'gpu_used':False,
              'artifact':str(_cpu_artifact)}
atomic_json(_cpu_artifact/'cpu_smoke.json',_cpu_summary)
print(_cpu_summary)
